A gente define o objetivo, se eh replicar o indice ou montar uma carteira equiponderada que perfomane melhor que o indice

### Imports

In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import re
import matplotlib.lines as mlines
import warnings
from IPython.display import display, Markdown
from pathlib import Path
import statsmodels.api as sm

warnings.filterwarnings("ignore")

### Global Variables

In [29]:
BASE_DIR = Path().resolve().parent
BRONZE_DIR = BASE_DIR / 'data' / 'bronze'
SILVER_DIR = BASE_DIR / 'data' / 'silver'
GOLD_DIR = BASE_DIR / 'data' / 'gold'

### Functions

In [30]:
def count_outliers_iqr(series):
    """Robust method based on quartiles"""
    if series.empty or not np.issubdtype(series.dtype, np.number): 
        return 0
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return ((series < (q1 - 1.5 * iqr)) | (series > (q3 + 1.5 * iqr))).sum()

def count_outliers_zscore(series, threshold=3):
    """Method based on standard deviation (Z-score)"""
    if series.empty or len(series) < 2 or not np.issubdtype(series.dtype, np.number): 
        return 0
    z_scores = np.abs(stats.zscore(series, nan_policy='omit'))
    return (z_scores > threshold).sum()

def is_valid_isin(isin):
    """Validates if the ISIN follows the international format"""
    if pd.isna(isin): 
        return False
    return bool(re.match(r"^[A-Z]{2}[A-Z0-9]{9}\d{1}$", str(isin)))

def _export_pickle(df, export_path, default_filename):
    """
    Helper for Parquet export (Silver/Gold layer best practice).
    """
    if not export_path: 
        return
        
    save_path = Path(export_path)
    if save_path.is_dir() or not save_path.suffix:
        save_path.mkdir(parents=True, exist_ok=True)
        save_path = save_path / f"{default_filename}.pickle"
        
    pd.to_pickle(df, save_path)
    print(f"Data saved to: {save_path}")


### Plots

In [31]:
def plot_full_diagnostic(asset_id, data_dict, quality_df, metadata_df):
    """
    Plots the NAV time series with all identified inconsistencies 
    and displays a summary quality table.
    """
    # 1. Data Retrieval
    try:
        # Match ID type to the index of quality_df (usually int or str)      
        fund_name = metadata_df.loc[asset_id]['name']
        print(data_dict)
        df = data_dict.get(asset_id).copy()
        print(df.head())
    except Exception as e:
        print(f"Error: Asset ID {asset_id} not found in the dataset. {e}")
        return

    # --- 2. Identifying Plot Markers ---
    nav = df['nav']
    
    # Outliers: IQR Method
    q1, q3 = nav.quantile(0.25), nav.quantile(0.75)
    iqr = q3 - q1
    iqr_outliers = df[(nav < (q1 - 1.5 * iqr)) | (nav > (q3 + 1.5 * iqr))]
    
    # Outliers: Z-Score Method (> 3 standard deviations)
    z_scores = np.abs(stats.zscore(nav, nan_policy='omit'))
    z_outliers = df[z_scores > 3]
    
    # Extreme Volatility (> 15% daily change)
    extreme_vol = df[nav.pct_change().abs() > 0.15]
    
    # Invalid Values (<= 0)
    invalid_vals = df[nav <= 0]

    # --- 3. Visualization ---
    plt.figure(figsize=(16, 7))
    
    # Handle Gaps for Visualization (Reindex to daily frequency)
    full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
    df_daily = df.reindex(full_range)
    df_interp = df_daily['nav'].interpolate(method='linear')
    
    # Plotting Lines
    plt.plot(df_interp.index, df_interp, color='red', linestyle='--', alpha=0.3, label='Gap Jump')
    plt.plot(df_daily.index, df_daily['nav'], color='royalblue', lw=2, label='Actual NAV', zorder=2)

    # Scatter Plotting Inconsistencies
    if not iqr_outliers.empty:
        plt.scatter(iqr_outliers.index, iqr_outliers['nav'], color='orange', 
                    label='IQR Outlier', s=50, alpha=0.7, zorder=3)
    
    if not z_outliers.empty:
        plt.scatter(z_outliers.index, z_outliers['nav'], color='red', marker='x', 
                    label='Z-Score Outlier (>3σ)', s=80, zorder=4)
        
    if not extreme_vol.empty:
        plt.scatter(extreme_vol.index, extreme_vol['nav'], edgecolors='black', 
                    facecolors='none', s=120, lw=1.5, label='Extreme Volatility (>15%)', zorder=5)

    if not invalid_vals.empty:
        plt.scatter(invalid_vals.index, invalid_vals['nav'], color='black', marker='s', 
                    label='Invalid Value (<=0)', s=100, zorder=6)

    # Formatting the Plot
    plt.title(f"Full Diagnostic: {fund_name} (ID: {asset_id})", fontsize=15, fontweight='bold', pad=20)
    plt.ylabel("NAV Value")
    plt.xlabel("Date")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)
    plt.tight_layout()
    plt.show()

    '''
    # --- 4. Summary Table Display ---
    display(Markdown(f"### Quality Summary Report: Asset {asset_id}"))
    
    # Select specific columns to show in the summary
    summary_cols = [
        'total_rows', 'gaps_data_%', 'outliers_zscore_%', 
        'stale_prices_%', 'invalid_values_%', 'extreme_volatility_%'
    ]
    
    # Transpose for better vertical readability
    summary_table = quality_df[summary_cols].T
    summary_table.columns = ['Current Value']
    
    # Add a 'Status' column based on thresholds (optional logic)
    display(summary_table.style.format("{:.2f}").background_gradient(cmap='YlOrRd', axis=0))
    '''

### Generate DFs

In [32]:
def process_fund_data(asset_dict, export_path=None):
    """
    Processes a dictionary of fund DataFrames to extract quality metrics, 
    metadata, and identify duplicates.
    
    Args:
        asset_dict (dict): Dictionary where keys are IDs and values are DataFrames.
        export_path (str/Path, optional): Directory to save the resulting CSVs.
        
    Returns:
        tuple: (df_quality, df_metadata, df_duplicated)
    """
    quality_list = []
    metadata_list = []
    all_duplicates_list = []

    print(f"Starting processing for {len(asset_dict)} assets...")

    for key, df in asset_dict.items():
        total_rows = len(df)
        if total_rows == 0:
            continue

        # 1. Data Standardization
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)

        # 2. Metadata Extraction
        af_id = df['allfunds_id'].iloc[0] if 'allfunds_id' in df.columns else "N/A"
        isin = df['isin'].iloc[0] if 'isin' in df.columns else "N/A"
        name = df['name'].iloc[0] if 'name' in df.columns else "N/A"
        
        metadata_list.append({'allfunds_id': af_id, 'isin': isin, 'name': name})

        # 3. Quality Diagnosis
        df.index = pd.to_datetime(df.index)
        df["nav"] = pd.to_numeric(df["nav"])
        
        nav_series = df['nav']
        diffs = nav_series.diff()
        returns = nav_series.pct_change().abs()
        
        count_iqr = count_outliers_iqr(nav_series)
        count_zscore = count_outliers_zscore(nav_series)
        
        expected_days = len(pd.bdate_range(start=df.index.min(), end=df.index.max()))
        count_gaps = max(0, expected_days - total_rows)
        
        count_stale = (diffs == 0).sum()
        count_invalid = (nav_series <= 0).sum()
        count_volat = (returns > 0.15).sum()
        
        skewness_val = nav_series.skew()

        quality_list.append({
            'source_key': key,
            'allfunds_id': af_id,
            'nav_nulls': nav_series.isnull().sum(),
            'first_index': df.index[0],
            'last_index': df.index[-1],
            'index_is_unique': df.index.is_unique,
            'index_duplicated_sum': df.index.duplicated().sum(),
            'total_rows': total_rows,
            
            # --- OUTLIER & DISTRIBUTION METRICS ---
            'outliers_iqr_abs': count_iqr,
            'outliers_iqr_%': round((count_iqr / total_rows) * 100, 2),
            'outliers_zscore_abs': count_zscore,
            'outliers_zscore_%': round((count_zscore / total_rows) * 100, 2),
            'nav_skewness': round(skewness_val, 4),
            
            # --- CONTINUITY & STALE DATA ---
            'gaps_data_%': round((count_gaps / expected_days) * 100, 2) if expected_days > 0 else 0,
            'stale_prices_%': round((count_stale / total_rows) * 100, 2),
            
            # --- INVALID VALUES ---
            'invalid_values_abs': count_invalid,
            'invalid_values_%': round((count_invalid / total_rows) * 100, 2),
            
            # --- RISK & METADATA ---
            'extreme_volatility_%': round((count_volat / total_rows) * 100, 2),
            'is_isin_valid': is_valid_isin(isin)
        })

        # 4. Duplicates Capture
        if df.index.duplicated().any():
            duplicates = df[df.index.duplicated(keep=False)].copy()
            duplicates['source_key'] = key
            all_duplicates_list.append(duplicates)

        # 5. In-place Cleaning (Modifies the input dictionary)
        asset_dict[key] = df.drop(columns=['isin', 'name'], errors='ignore')

    # --- CONSOLIDATION ---
    df_quality = pd.DataFrame(quality_list).set_index('allfunds_id')
    df_metadata = pd.DataFrame(metadata_list).drop_duplicates(subset=['allfunds_id']).set_index('allfunds_id')
    df_duplicated = pd.concat(all_duplicates_list) if all_duplicates_list else pd.DataFrame()

    # --- OPTIONAL EXPORT ---
    if export_path:
        path = Path(export_path)
        path.mkdir(parents=True, exist_ok=True)
        df_quality.to_csv(path / 'quality_data.csv')
        df_metadata.to_csv(path / 'metadata.csv')
        if not df_duplicated.empty:
            df_duplicated.to_csv(path / 'duplicated_rows.csv')
        print(f"Files successfully saved to: {export_path}")

    return df_quality, df_metadata, df_duplicated


### Filtering Data

In [33]:
def filter_assets_by_quality(df_quality, upper_q=0.80, lower_q=0.20, date_q=0.95, export_path=None):
    """
    Filters assets based on statistical quality thresholds calculated from quantiles.
    """
    
    # 1. Dynamic Threshold Calculation
    thresholds = {
        'min_rows': df_quality['total_rows'].quantile(lower_q),
        'cutoff_date': df_quality['first_index'].quantile(date_q),
        'max_gaps': df_quality['gaps_data_%'].quantile(upper_q),
        'max_outliers': df_quality['outliers_zscore_%'].quantile(upper_q),
        'max_stale': df_quality['stale_prices_%'].quantile(upper_q),
        'max_volatility': df_quality['extreme_volatility_%'].quantile(upper_q),
        'max_skewness': df_quality['nav_skewness'].abs().quantile(upper_q)
    }

    print("--- Calculated Statistical Thresholds ---")
    for metric, value in thresholds.items():
        print(f"{metric}: {value}")

    # 2. Applying Filter Criteria
    # Fixed the invalid_values logic and added negative_prices constraint
    conditions = (
        (df_quality['index_is_unique'] == True) &
        (df_quality['index_duplicated_sum'] == 0) &
        (df_quality['total_rows'] >= thresholds['min_rows']) &
        (df_quality['first_index'] <= thresholds['cutoff_date']) &
        (df_quality['outliers_zscore_%'] <= thresholds['max_outliers']) &
        (df_quality['gaps_data_%'] <= thresholds['max_gaps']) &
        (df_quality['stale_prices_%'] <= thresholds['max_stale']) &
        (df_quality['invalid_values_%'] == 0) &
        (df_quality['nav_skewness'].abs() <= thresholds['max_skewness']) &
        (df_quality['extreme_volatility_%'] <= thresholds['max_volatility'])
    )

    df_filtered = df_quality.loc[conditions].copy()

    # 3. Summary Results
    total = len(df_quality)
    approved = len(df_filtered)
    pct = round((approved / total) * 100, 2) if total > 0 else 0
    
    print(f"\nFiltering Result: {approved} / {total} assets approved ({pct}%).")

    if export_path:
        save_path = Path(export_path)
        if save_path.is_dir() or not save_path.suffix:
            save_path.mkdir(parents=True, exist_ok=True)
            save_path = save_path / f'quality_data_filtered.csv'
            
        df_filtered.to_csv(save_path)
        print(f"Filtered report saved to: {save_path}")

    return df_filtered


def filter_raw_data_by_quality(raw_data, filtered_df, export_path=None):
    """
    Filters the main data dictionary to retain only the assets that passed 
    the quality filters and optionally saves the results to the Silver layer.
    
    Args:
        raw_data (dict): The original dictionary {source_key: DataFrame}.
        filtered_df (pd.DataFrame): The DataFrame containing approved assets.
        export_path (str/Path, optional): Directory to save the filtered data.
        
    Returns:
        dict: A new dictionary containing only high-quality assets.
    """
    # 1. Get the list of approved source keys
    # Using the 'source_key' column derived during the processing phase
    approved_keys = filtered_df['source_key'].unique()
    
    # 2. Create the filtered dictionary using dictionary comprehension
    filtered_data = {
        key: raw_data[key] 
        for key in approved_keys 
        if key in raw_data
    }
    
    # 3. Summary of memory optimization
    original_count = len(raw_data)
    final_count = len(filtered_data)
    removed_count = original_count - final_count
    
    print(f"--- Data Pruning Summary ---")
    print(f"Assets before pruning: {original_count}")
    print(f"Assets after pruning:  {final_count}")
    print(f"Removed (low quality): {removed_count} assets")
    print("-" * 30)
    
    # 4. Export Logic (The part we are adding)
    if export_path:
        _export_pickle(filtered_data, export_path, 'filtered_navs')
        
    return filtered_data

def fill_time_series_gaps(data_dict, export_path=None):
    """
    Expands each DataFrame to include all calendar days between its 
    start and end dates, filling missing NAV values using forward fill.
    """
    processed_data = {}
    
    print(f"Starting gap filling for {len(data_dict)} assets...")

    for key, df in data_dict.items():
        # 1. Define the full range of calendar days
        full_business_range = pd.date_range(
            start=df.index.min(), 
            end=df.index.max(), 
            freq='B'
        )
        
        # 2. Reindex the DataFrame
        # This inserts rows with NaN for any missing dates in the range
        df_filled = df.reindex(full_business_range)
        
        # 3. Fill the NAV column using Forward Fill (ffill)
        # We use ffill because today's price is assumed to be yesterday's 
        # if no new data is available.
        df_filled['nav'] = df_filled['nav'].ffill()
        
        # 4. Fill metadata columns
        # Since 'allfunds_id' is constant for the whole DF, we fill it too
        if 'allfunds_id' in df_filled.columns:
            df_filled['allfunds_id'] = df_filled['allfunds_id'].ffill()
            
        processed_data[key] = df_filled

    print("Gap filling completed successfully.")
    
    # 4. Export logic
    if export_path:
        _export_pickle(processed_data, export_path, 'filtered_navs_no_gaps')

    return processed_data



### Generate features

In [ ]:

def calculate_monthly_returns_dict(data_dict, export_path=None):
    """
    Calculates monthly returns for each fund and returns a dictionary of DataFrames.
    TESTING MODE: Only processes the first 10 funds.
    """
    returns_dict = {}

    print(f"Processing monthly returns for {len(data_dict)}")
    
    for asset_id, df in data_dict.items():
        # 1. Resample to Month End (ME) to get the last price of the month
        monthly_nav = df['nav'].resample('ME').last()
        
        # 2. Calculate Log Returns
        log_returns = np.log(monthly_nav).diff().dropna().to_frame()
        log_returns.columns = ['returns']
        
        returns_dict[asset_id] = log_returns
    
    # Export logic
    if export_path:
        # Consolidate dictionary to a Wide DataFrame for Parquet export
        temp_wide = pd.concat({k: v['returns'] for k, v in returns_dict.items()}, axis=1)
        _export_pickle(temp_wide, export_path, 'monthly_log_returns')
        
    return returns_dict

def get_fama_french_factors(csv_path, export_path=None):
    """
    Reads daily Fama-French factors and converts them to monthly frequency.
    Note: The user file is '3 Factors Daily' (Mkt-RF, SMB, HML).
    """
    # 1. Load data (Daily format: YYYYMMDD)
    factors = pd.read_csv(csv_path)
    
    # 2. Convert 8-digit integer date to datetime
    factors['Date'] = pd.to_datetime(factors['Date'], format='%Y%m%d')
    factors.set_index('Date', inplace=True)
    
    # 3. Resample from Daily to Monthly to match fund returns
    # Factors are returns, so we compound them for the month
    # Formula: [(1 + d1)*(1 + d2)...] - 1
    factors_monthly = (1 + factors/100).resample('ME').prod() - 1
    
    if export_path:
        _export_pickle(factors_monthly, export_path, 'fama_french_factors_monthly')
        
    return factors_monthly

def extract_fund_features_from_dict(returns_dict, factors_df, export_path=None):
    """
    Runs OLS regression using only 3 factors (Mkt-RF, SMB, HML).
    """
    features_list = []
    
    print(f"Running regressions for {len(returns_dict)} assets...")

    for asset_id, fund_ret_df in returns_dict.items():
        # Align dates (Monthly vs Monthly)
        common_dates = fund_ret_df.index.intersection(factors_df.index)
        
        if len(common_dates) < 6: # Minimum data requirement
            continue
            
        # Dependent Variable: Excess Return
        y = fund_ret_df.loc[common_dates, 'returns'] - factors_df.loc[common_dates, 'RF']
        
        # Independent Variables: ONLY 3 factors available in your file
        X = sm.add_constant(factors_df.loc[common_dates, ['Mkt-RF', 'SMB', 'HML']])
        
        model = sm.OLS(y, X, missing='drop').fit()
        
        features_list.append({
            'allfunds_id': asset_id,
            'alpha': model.params['const'],
            'beta_mkt': model.params['Mkt-RF'],
            'beta_smb': model.params['SMB'],
            'beta_hml': model.params['HML'],
            'r_squared': model.rsquared
        })
    
    # Safety check: if no regressions were successful, don't try to set_index
    if not features_list:
        print("Error: No common dates found between funds and factors. Check your date ranges.")
        return pd.DataFrame()

    df_features = pd.DataFrame(features_list).set_index('allfunds_id')
    
    if export_path:
        _export_pickle(df_features, export_path, 'funds_features')
    
    return df_features

def prepare_feature_matrix(returns_dict, factors_df, export_path):
    """
    Calculates quantitative variables and prepares the feature matrix for clustering.
    Aligned with the assignment's requirement to extract relevant financial features.
    """
    features_list = []
    
    print(f"Preparing feature matrix for {len(returns_dict)} assets...")

    for asset_id, fund_ret_df in returns_dict.items():
        # 1. Temporal Alignment
        common_dates = fund_ret_df.index.intersection(factors_df.index)
        if len(common_dates) < 12: continue # Minimum 1 year of data
            
        y = fund_ret_df.loc[common_dates, 'returns'] - factors_df.loc[common_dates, 'RF']
        X = sm.add_constant(factors_df.loc[common_dates, ['Mkt-RF', 'SMB', 'HML']])
        
        # 2. Regression (Style Analysis)
        model = sm.OLS(y, X, missing='drop').fit()
        
        # 3. Risk & Performance Metrics (Annualized)
        volatility = fund_ret_df['returns'].std() * np.sqrt(12)
        mean_excess_return = y.mean() * 12
        sharpe_ratio = mean_excess_return / volatility if volatility != 0 else 0
        
        features_list.append({
            'allfunds_id': asset_id,
            'alpha': model.params['const'],
            'beta_mkt': model.params['Mkt-RF'],
            'beta_smb': model.params['SMB'],
            'beta_hml': model.params['HML'],
            'r_squared': model.rsquared,
            'volatility': volatility,
            'sharpe_ratio': sharpe_ratio
        })
        
    # Create the final Feature Matrix
    feature_matrix = pd.DataFrame(features_list).set_index('allfunds_id')
    
    if export_path:
        _export_pickle(feature_matrix, export_path, 'feature_matrix')
    
    return feature_matrix

### Filter funds with Asia Bias

In [ ]:
def filter_asia_bias_funds(df_features, data_dict, beta_min=0.5, r2_min=0.3, export_path=None):
    """
    Filters funds that demonstrate a clear Asian market bias based on 
    regression results (Beta and R-squared).
    """
    # 1. Apply filters to the feature matrix
    # We look for funds with significant exposure (Beta) and high fit (R2)
    asia_bias_mask = (df_features['beta_mkt'] > beta_min) & (df_features['r_squared'] > r2_min)
    
    filtered_features = df_features[asia_bias_mask].copy()
    
    # 2. Get the IDs of the approved funds
    approved_ids = filtered_features.index.tolist()
    
    # 3. Create a new dictionary containing only the approved funds
    filtered_data_dict = {asset_id: data_dict[asset_id] for asset_id in approved_ids if asset_id in data_dict}
    
    print(f"--- Asia Bias Selection Summary ---")
    print(f"Funds analyzed: {len(df_features)}")
    print(f"Funds with clear Asia bias: {len(approved_ids)}")
    print(f"Filter Criteria: Beta Mkt > {beta_min} and R-squared > {r2_min}")
    print("-" * 35)
    
    # 4. Export logic
    if export_path:
        _export_pickle(filtered_data_dict, export_path, 'asia_bias_funds')
        _export_pickle(filtered_features, export_path, 'features_asia_bias_funds')
    
    return filtered_data_dict, filtered_features

### Main

In [ ]:
def main():
    # 1. Load the raw data
    pickle_path = BRONZE_DIR / 'navs.pickle'
    
    if not pickle_path.exists():
        print(f"Error: {pickle_path} not found.")
        return

    raw_data = pd.read_pickle(pickle_path)
    #raw_data = dict(list(raw_data.items())[:10])
    asset_ids = list(raw_data.keys())
    
    print(f"Loaded {len(asset_ids)} assets from Bronze layer.")
    
    # 2. Run Processing (The function we created earlier)
    df_quality, df_metadata, df_duplicated = process_fund_data(
        raw_data, 
        export_path=SILVER_DIR
    )
    
    # 3. Run Filtering
    '''
    uppers = [0.75,0.8,0.85,0.9,0.95]
    lowers = [0.05,0.1,0.15,0.2,0.25]
    
    for upper in uppers:
        for lower in lowers:
            df_filtered = filter_assets_by_quality(
                df_quality, 
                upper_q=upper, 
                lower_q=lower, 
                export_path=SILVER_DIR
            )
    '''
    df_filtered = filter_assets_by_quality(
        df_quality, 
        upper_q=0.80, 
        lower_q=0.2, 
        export_path=SILVER_DIR
    )
    
    # 4. Run Filtering raw data by quality
    data = filter_raw_data_by_quality(
        raw_data, 
        df_filtered, 
        export_path=SILVER_DIR
    )
        
    # 5. Run Fill Time Series with Gaps
    processed_data = fill_time_series_gaps(data, SILVER_DIR)
    
    # 6. Generate Monthly Log Returns
    monthly_returns = calculate_monthly_returns_dict(processed_data, SILVER_DIR)
    
    # 7. Read Fama French Factors
    csv_path = BRONZE_DIR / 'Asia_Pacific_ex_Japan_3_Factors_Daily.csv'
    factors_df = get_fama_french_factors(csv_path, SILVER_DIR)
    
    # 8. Generate DF Features
    df_features = extract_fund_features_from_dict(monthly_returns, factors_df, SILVER_DIR)
    
    # 9. Feature Matrix
    feature_matrix = prepare_feature_matrix(monthly_returns, factors_df, SILVER_DIR)
    
    # 10. Filter funds with Asia bias
    asia_bias_funds, asia_bias_funds_features = filter_asia_bias_funds(df_features, data, SILVER_DIR)
    
    return raw_data, data, df_metadata, df_quality, df_filtered, processed_data, monthly_returns, factors_df, df_features, feature_matrix, asia_bias_funds, asia_bias_funds_features

# --- EXECUTION ---
if __name__ == "__main__":
    raw_data, data, df_metadata, df_quality, df_filtered, processed_data, monthly_returns, factors_df, df_features, feature_matrix, asia_bias_funds, asia_bias_funds_features = main()

Loaded 24822 assets from Bronze layer.
Starting processing for 24822 assets...
Files successfully saved to: C:\Users\piett\OneDrive\Desktop\Pietro\Master MIAX\Clases\3.Inteligencia Artificial Basica\Taller\Supervisado\data\silver
--- Calculated Statistical Thresholds ---
min_rows: 898.0
cutoff_date: 2019-06-14 00:00:00
max_gaps: 6.82
max_outliers: 0.4
max_stale: 5.22
max_volatility: 0.0
max_skewness: 0.8880800000000003

Filtering Result: 9002 / 24822 assets approved (36.27%).
Filtered report saved to: C:\Users\piett\OneDrive\Desktop\Pietro\Master MIAX\Clases\3.Inteligencia Artificial Basica\Taller\Supervisado\data\silver\quality_data_filtered.csv
--- Data Pruning Summary ---
Assets before pruning: 24822
Assets after pruning:  9002
Removed (low quality): 15820 assets
------------------------------
Data saved to: C:\Users\piett\OneDrive\Desktop\Pietro\Master MIAX\Clases\3.Inteligencia Artificial Basica\Taller\Supervisado\data\silver\filtered_navs.pickle
Starting gap filling for 9002 asse